Nuclei Segmenting Following Normal Cellpose Analysis : Assumes cell/RNA/nuclei channels have been split and saved in directory structure as in 'SplitAndRescale3'

In [1]:
import os
import numpy as np
import pandas as pd
from skimage.transform import rescale
from skimage.io import imread, imsave, imshow
from skimage.filters import gaussian
from tkinter import Tk
from tkinter import filedialog
import tifffile as tf
import dask
from skimage import io
import cv2 as cv

dask.config.set({"array.slicing.split_large_chunks": False})
#fTypes = (".tif", ".czi")
fTypes = (".tif")


import matplotlib
import pyclesperanto_prototype as cle
cle.select_device("Apple M1 Max")

<Apple M1 Max on Platform: Apple (2 refs)>

Create file structure (similar to cell files in OG code)

In [5]:
root = Tk()
root.withdraw()
dirName = filedialog.askdirectory()

nucPath = dirName + "/nuc"
nfiles = os.listdir(nucPath)
dsPath = nucPath + "/downscaled"
nlabPath = dirName + "/nuc/nLabs"

nfiles = os.listdir(nucPath)

2024-05-05 19:46:39.056 python[16242:30658058] +[CATransaction synchronize] called within transaction


Downscale nuclei files 

In [4]:
root = Tk()
root.withdraw()
dirName = filedialog.askdirectory()

nucPath = dirName + "/nuc"
nfiles = os.listdir(nucPath)
dsPath = nucPath + "/downscaled"
nlabPath = dirName + "/nuc/nLabs"

if not os.path.exists(dsPath):
    os.mkdir(dsPath)
if not os.path.exists(nlabPath):
    os.mkdir(nlabPath)

nfiles = os.listdir(nucPath)
for filename in (nfiles):
    if filename.endswith(".tif") and not filename.startswith("."):
        print(filename)
        path = os.path.join(nucPath, filename)
        img = imread(path) 
        ds_nuc = rescale(img, (1, 0.25, 0.25), anti_aliasing=True)
        dsG = gaussian(ds_nuc, sigma=(0,1.3, 1.3), truncate = 3.5)
        nfile = "DS_" + filename
        outpath = os.path.join(dsPath,nfile)
        if not os.path.exists(outpath):
            imsave(outpath, dsG, check_contrast=False)

2024-05-05 19:27:01.128 python[15899:30605596] +[CATransaction synchronize] called within transaction
2024-05-05 19:27:01.167 python[15899:30605596] +[CATransaction synchronize] called within transaction
2024-05-05 19:27:02.906 python[15899:30605596] +[CATransaction synchronize] called within transaction


nuc_112023_K913P20_A3_LH_40xstack_T2b_K9.tif
nuc_112023_K913P20_A5_LH_40xstack_T2b_K9_fine.tif
nuc_112023_K913P20_A6_LH_40xstack_T2b_K9_good.tif
nuc_120823_K913_A1_RH_D3_K9.tif
nuc_120823_K913_A2_RH_D3_K9.tif
nuc_120823_K913_A3_RH_D3_K9.tif
nuc_120823_K913_A4_RH_D3_K9.tif
nuc_120823_K913_A5_RH_D3_K9.tif
nuc_120823_K913_B1_RHup_D3_K9.tif
nuc_120823_K913_B2_RHup_D3_K9.tif
nuc_120823_K913_B3_LHup_D3_K9.tif
nuc_120823_K913_B4_LHup_D3_K9.tif
nuc_120823_K913_B5_RHdw_D3_K9.tif
nuc_120823_K913_B6_RHdw_D3_K9.tif
nuc_120823_K913_B7_LHdw_D3_K9.tif
nuc_120823_K913_B8_LHdw_D3_K9.tif
nuc_120823_K913P20_B1_RHTop_D3_K9.tif
nuc_120823_K913P20_B1_RHTop_T2b_K9.tif
nuc_120823_K913P20_B2_LHTop_T2b_K9.tif
nuc_120823_K913P20_B3_RHTop_T2b_K9.tif
nuc_120823_K913P20_B4_LHTop_T2b_K9.tif
nuc_120823_K913P20_B5_RHdw_T2b_K9.tif
nuc_120823_K913P20_B6_LHdw_T2b_K9.tif
nuc_120823_K913P20_B7_LHdw_T2b_K9.tif
nuc_120823_K913P20_B8_LHdw_T2b_K9.tif
nuc_020624_K913_D5_RH_D3_K9.tif
nuc_020624_K913_D6_RH_D3_K9.tif
nuc_020624_K9

In [1]:
from cellpose import models
from cellpose import io as cpio

#cellPath = "/Users/sachanelson/programming/Jupyter/FISH/dataFromLiwei/cell"
#use_GPU = core.use_gpu()
#print('>>> GPU activated? %d'%use_GPU)

# call logger_setup to have output of cellpose written
from cellpose.io import logger_setup
logger_setup();

model = models.Cellpose(gpu=False, model_type='cyto')
channels = [0,0]


2024-05-05 19:45:47,018 [INFO] WRITING LOG OUTPUT TO /Users/nelsonlab/.cellpose/run.log
2024-05-05 19:45:47,019 [INFO] 
cellpose version: 	2.2.3 
platform:       	darwin 
python version: 	3.9.18 
torch version:  	2.1.2.post2
2024-05-05 19:45:47,019 [INFO] >>>> using CPU
2024-05-05 19:45:47,019 [INFO] >> cyto << model set to be used
2024-05-05 19:45:47,020 [INFO] WARNING: MKL version on torch not working/installed - CPU version will be slightly slower.
2024-05-05 19:45:47,020 [INFO] see https://pytorch.org/docs/stable/backends.html?highlight=mkl
2024-05-05 19:45:47,084 [INFO] >>>> model diam_mean =  30.000 (ROIs rescaled to this size during training)


In [ ]:
Run Cellpose on Nuclei: May want to optimize stitch_threshold and cellprob_threshold in GUI

In [6]:
nfiles2 = os.listdir(dsPath)
for filename in (nfiles2):
   if filename.endswith(".tif") and not filename.startswith("."):
      fpath = os.path.join(dsPath, filename)
      outpath = os.path.join(nlabPath, filename)
      print("reading file",fpath)
      img = cpio.imread(fpath)
# might need to play with diameter, pre measure, or use the diameter=None setting
      masks, flows, styles, diams = model.eval(img, diameter=40, do_3D=False, stitch_threshold=0.2,cellprob_threshold=-0.5,min_size=20)
      print("ran model")
#      fpath2 = cellPath + "/masks"
#      if not os.path.exists(fpath2):  
#         os.mkdir(fpath2)  
      cpio.save_masks(img, masks, flows, outpath, png=False, tif=True, save_flows=True)
   else:
      continue

reading file /Volumes/RWK_Imaging/K913_nucquant/nuc/downscaled/DS_nuc_112023_K913P20_A3_LH_40xstack_T2b_K9.tif
2024-05-05 19:47:44,894 [INFO] reading tiff with 19 planes


100%|███████████████████████████████████████████| 19/19 [00:00<00:00, 23.47it/s]

2024-05-05 19:47:45,739 [INFO] ~~~ FINDING MASKS ~~~
2024-05-05 19:47:45,784 [INFO] multi-stack tiff read in as having 19 planes 1 channels


2024-05-05 19:48:17,385 [INFO] 100%|##########| 19/19 [00:31<00:00,  1.66s/it]
2024-05-05 19:48:17,386 [INFO] network run in 31.58s
2024-05-05 19:48:29,644 [INFO] stitching 19 planes using stitch_threshold=0.200 to make 3D masks
2024-05-05 19:48:30,221 [INFO] masks created in 12.83s
2024-05-05 19:48:30,557 [INFO] >>>> TOTAL TIME 44.82 sec
ran model
reading file /Volumes/RWK_Imaging/K913_nucquant/nuc/downscaled/DS_nuc_112023_K913P20_A5_LH_40xstack_T2b_K9_fine.tif
2024-05-05 19:48:31,190 [INFO] reading tiff with 22 planes


100%|███████████████████████████████████████████| 22/22 [00:00<00:00, 24.84it/s]

...

2024-05-05 20:26:14,913 [INFO] ~~~ FINDING MASKS ~~~
2024-05-05 20:26:14,953 [INFO] multi-stack tiff read in as having 16 planes 1 channels
2024-05-05 20:26:14,973 [INFO] 0%|          | 0/16 [00:00<?, ?it/s]


2024-05-05 20:26:43,529 [INFO] 100%|##########| 16/16 [00:28<00:00,  1.78s/it]
2024-05-05 20:26:43,530 [INFO] network run in 28.56s
2024-05-05 20:26:51,479 [INFO] stitching 16 planes using stitch_threshold=0.200 to make 3D masks
2024-05-05 20:26:51,645 [INFO] masks created in 8.11s
2024-05-05 20:26:51,925 [INFO] >>>> TOTAL TIME 37.01 sec
ran model
reading file /Volumes/RWK_Imaging/K913_nucquant/nuc/downscaled/DS_nuc_022624_K913_A4_RH_T2b_K9.tif
2024-05-05 20:26:52,454 [INFO] reading tiff with 18 planes


100%|███████████████████████████████████████████| 18/18 [00:00<00:00, 23.51it/s]

2024-05-05 20:26:53,244 [INFO] ~~~ FINDING MASKS ~~~


2024-05-05 20:26:53,285 [INFO] multi-stack tiff read in as having 18 planes 1 channels
2024-05-05 20:27:25,323 [INFO] 100%|##########| 18/18 [00:32<00:00,  1.78s/it]
2024-05-05 20:27:25,324 [INFO] network run in 32.02s
2024-05-05 20:27:34,697 [INFO] stitching 18 planes using stitch_threshold=0.200 to make 3D masks
2024-05-05 20:27:34,830 [INFO] masks created in 9.51s
2024-05-05 20:27:35,139 [INFO] >>>> TOTAL TIME 41.89 sec
ran model


ONLY upscale and Expand Expand labels: Do this if you aren't segmenting cells too!

In [12]:
##ONLY Upscale and Expand nuclei labels
from skimage.segmentation import clear_border, expand_labels
from napari_simpleitk_image_processing import label_statistics
from skimage.measure import regionprops_table

#Select pixel radius to expand nuclei labels by in X-Y (upscaled): Default 10
expfact=10

#By default expands labels in +/- 1 z plane 

nucPath = dirName + "/nuc"
dsPath = nucPath + "/downscaled"
nlabPath = nucPath + "/nLabs"
upPath = os.path.join(nlabPath,"upscaled_expanded")
if not os.path.exists(upPath):
    os.mkdir(upPath)

#Main loop - load in label/img files, filter out bad calls, save cell (img) properties, upscale label file
for filename in os.listdir(dsPath):
   if filename.endswith(".tif") and not filename.startswith("."):
       print(filename)
       filePath = os.path.join(dsPath, filename)
       labName = filename.replace(".tif","_cp_masks.tif")
       labPath = os.path.join(nlabPath, labName)
       if os.path.exists(labPath):
           lab = io.imread(labPath)
           if labName.endswith("_cp_masks.tif") and labName.startswith("DS"):
               lab_up = rescale(lab, (1, 4, 4), anti_aliasing=False, preserve_range=True, order=0)
               c4fname = "exp" + str(expfact) "_" + labName[3:]
               c4fpath = os.path.join(upPath, c4fname)
               labex = expand_labels(lab_up,1)
               labey = labex.copy()
               for i in range(labex.shape[0]):
                   labey[i,:,:] = expand_labels(labex[i,:,:],expfact)
               imsave(c4fpath, labey, check_contrast=False)
               print("saved rescaled, expanded files")
           else:
               print ("wrong filetype--not DS cp_mask", filename)
               continue
       else:
           print("label file not found")
           continue



DS_nuc_112023_K913P20_A5_LH_40xstack_T2b_K9_fine.tif
saved rescaled, expanded files


Upscale and Expand Nuclei labels, get intensity (from GFP channel), identify nuclei/cell label overlap
Only possible if you ran pipeline on cell channel!

In [104]:
#Upscale and expand nuclei labels, find overlapping cells, create label stats (size, GFP intensity)

from scipy import stats
from skimage.segmentation import clear_border, expand_labels
from napari_simpleitk_image_processing import label_statistics
from skimage.measure import regionprops_table

root = Tk()
root.withdraw()
strtPath = "/Users/nelsonlab/"
dirName = filedialog.askdirectory(initialdir=strtPath)

#print(imgFile, "type=", type(imgFile))
cellFold = dirName + "/cell"
labFold = os.path.join(cellFold,"cLabs/upscaled")
orCellFold = os.path.join(cellFold,"origSize")

nucPath = dirName + "/nuc"
dsPath = nucPath + "/downscaled"
nlabPath = nucPath + "/nLabs"
upPath = os.path.join(nlabPath,"upscaled_expanded")

if not os.path.exists(upPath):  
# Create directory for storing the tiff files for cell channels 
   os.mkdir(upPath)
    
#Select pixel radius to expand nuclei labels by in X-Y (upscaled): Default 10
expfact=10

#Main loop - load in label/img files, filter out bad calls, save cell (img) properties, upscale label file
for filename in os.listdir(dsPath):
   if filename.endswith(".tif") and not filename.startswith("."):
       print(filename)
       filePath = os.path.join(dsPath, filename)
       labName = filename.replace(".tif","_cp_masks.tif")
       labPath = os.path.join(nlabPath, labName)
       if os.path.exists(labPath):
           #theimg = io.imread(filePath)
           lab = io.imread(labPath)
           #labfilt = lab.copy()
           #Find and filter false positives (not sure how to do on nuclei yet...)
           #regionprops = label_statistics(theimg, lab, True, True, True, True)
           #cellidx = regionprops[regionprops["sum"] < cutoff]["label"].to_numpy()
           #labfilt[np.where(np.isin(labfilt, cellidx))] = 0
           
           if labName.endswith("_cp_masks.tif") and labName.startswith("DS"):
               lab_up = rescale(lab, (1, 4, 4), anti_aliasing=False, preserve_range=True, order=0)
               prefix = "exp" + str(expfact) + "_" 
               c4fname = prefix + labName[3:]
               c4fpath = os.path.join(upPath, c4fname)

               labex = expand_labels(lab_up,1)
               labey = labex.copy()
               for i in range(labex.shape[0]):
                   labey[i,:,:] = expand_labels(labex[i,:,:],expfact)
               imsave(c4fpath, labey, check_contrast=False)
               print("saved rescaled, expanded files")

               #Load in cell label labels/upscaled cell images
               labCellfile = c4fname.replace(prefix + "nuc","C1")
               labCellpath = os.path.join(labFold,labCellfile)
               
               orCellfile = labCellfile.replace("_cp_masks.tif",".tif")
               orCellpath = os.path.join(orCellFold,orCellfile)
               
               orCell = imread(orCellpath)
               labCell = imread(labCellpath)

               #Identify nuclei with >=50% overlap with GFP+ cell 
               labfilt = lab_up.copy()
               labfilt[labCell==0] = 0 
               

               ind,cnt = np.unique(lab_up,return_counts=True)
               ind2,cnt2 = np.unique(labfilt,return_counts=True)
               cellnuc = ind2[cnt2/cnt[np.isin(ind,ind2)] > 0.5]

               #Which cell overlaps?
               testarray = np.full(len(np.unique(labey)),0)
               for p in cellnuc:
                   if p!=0:
                       vals, counts = np.unique(labCell[lab_up==p], return_counts=True)
                       mode_value = vals[np.argwhere(counts == np.max(counts))]
                       testarray[p] = mode_value
                   
               #summarize properties!    
               cLabProps = regionprops_table(labey, orCell, properties=("label","centroid", "area", "intensity_max", "intensity_mean", "num_pixels"))           
               dfCellProps = pd.DataFrame(cLabProps)
               dfCellProps["isCell"] = np.isin(dfCellProps["label"],cellnuc)
               dfCellProps["cellLab"] = testarray[1:]
               dfCellProps.to_csv(os.path.join(upPath,os.path.basename(os.path.splitext(filePath)[0])[3:] + "_nucprops" + str(expfact) + ".csv"))
               print("saved filtered label statistics")
           else:
               print ("wrong filetype--not DS cp_mask", filename)
               continue
       else:
           print("label file not found")
           continue



2024-05-08 16:36:31.325 python[16242:30658058] +[CATransaction synchronize] called within transaction
2024-05-08 16:36:31.390 python[16242:30658058] +[CATransaction synchronize] called within transaction
2024-05-08 16:36:32.848 python[16242:30658058] +[CATransaction synchronize] called within transaction


DS_nuc_112023_K913P20_A3_LH_40xstack_T2b_K9.tif
saved rescaled, expanded files


/var/folders/05/0lb791q54mq3py_0zyc6cp080000gp/T/ipykernel_16242/3070274068.py:80: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  testarray[p] = mode_value


saved filtered label statistics
DS_nuc_112023_K913P20_A5_LH_40xstack_T2b_K9_fine.tif
saved rescaled, expanded files


/var/folders/05/0lb791q54mq3py_0zyc6cp080000gp/T/ipykernel_16242/3070274068.py:80: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  testarray[p] = mode_value


...

saved filtered label statistics
DS_nuc_120823_K913P20_B7_LHdw_T2b_K9.tif
saved rescaled, expanded files


/var/folders/05/0lb791q54mq3py_0zyc6cp080000gp/T/ipykernel_16242/3070274068.py:80: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  testarray[p] = mode_value


saved filtered label statistics
DS_nuc_120823_K913P20_B8_LHdw_T2b_K9.tif
saved rescaled, expanded files


/var/folders/05/0lb791q54mq3py_0zyc6cp080000gp/T/ipykernel_16242/3070274068.py:80: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  testarray[p] = mode_value


saved filtered label statistics


Prepare for RNA puncta/nuceli overlap quantification

In [105]:
import glob

root = Tk()
root.withdraw()
strtPath = "/Users/nelsonlab/"
dirName = filedialog.askdirectory(initialdir=strtPath)
#files containing upscaled, expanded nuclei labels
nucPath = dirName + "/nuc"
nlabPath = nucPath + "/nLabs/upscaled_expanded"

# this subfolder has the RNA label files
rLabFold = dirName + "/rna/rLabs"                                
 
tabFold = dirName + "/table"
if not os.path.exists(tabFold):  
   os.mkdir(tabFold)  
    
rLabFiles = os.listdir(rLabFold)
nfiles = os.listdir(nlabPath)
print("nucFold=",nlabPath)
print(nfiles)

2024-05-08 17:12:38.808 python[16242:30658058] +[CATransaction synchronize] called within transaction
2024-05-08 17:12:38.843 python[16242:30658058] +[CATransaction synchronize] called within transaction
2024-05-08 17:12:40.416 python[16242:30658058] +[CATransaction synchronize] called within transaction


nucFold= /Volumes/RWK_Imaging/K913_nucquant/nuc/nLabs/upscaled_expanded
['exp20_nuc_021524_K913_B1_RHtop_T2b_K9_cp_masks.tif', 'nuc_021524_K913_B1_RHtop_T2b_K9_nucprops.csv', 'exp20_nuc_112023_K913P20_A5_LH_40xstack_T2b_K9_fine_cp_masks.tif', 'exp20_nuc_112023_K913P20_A3_LH_40xstack_T2b_K9_cp_masks.tif', 'nuc_112023_K913P20_A3_LH_40xstack_T2b_K9_nucprops.csv', '._nuc_112023_K913P20_A3_LH_40xstack_T2b_K9_nucprops.csv', 'nuc_112023_K913P20_A5_LH_40xstack_T2b_K9_fine_nucprops.csv', 'exp20_nuc_112023_K913P20_A6_LH_40xstack_T2b_K9_good_cp_masks.tif', 'nuc_112023_K913P20_A6_LH_40xstack_T2b_K9_good_nucprops.csv', 'exp20_nuc_120823_K913_A1_RH_D3_K9_cp_masks.tif', 'nuc_120823_K913_A1_RH_D3_K9_nucprops.csv', 'exp20_nuc_120823_K913_A2_RH_D3_K9_cp_masks.tif', 'nuc_120823_K913_A2_RH_D3_K9_nucprops.csv', 'exp20_nuc_120823_K913_A3_RH_D3_K9_cp_masks.tif', 'nuc_120823_K913_A3_RH_D3_K9_nucprops.csv', 'exp20_nuc_120823_K913_A4_RH_D3_K9_cp_masks.tif', 'nuc_120823_K913_A4_RH_D3_K9_nucprops.csv', 'exp20_nuc

Measure overlap of Nuclei/RNA - outputs ALL puncta/Cell overlap, punta size, % Overlap
Use upscaled, expanded nuclei for this measurement

In [78]:
import re
verbose = False

#rTabFiles = os.listdir(tabFold)
#intersections = [rf for rf in rTabFiles if rf.endswith("intersections.csv")]
#res = [fi.replace('intersections.csv', 'cp_masks.tif') for fi in intersections]
#res2 = [re.sub('rna[0-9]','C1',fi) for fi in res] 

for filename in nfiles:
    if filename.endswith("_cp_masks.tif") and filename.startswith("exp" + str(expfact)):
        #comment out if you don't want to reprocess files you've already done 
        #if filename in res2: continue
        print (filename)
        filename_rna = filename.replace("_cp_masks.tif","_lab.tif")
        nFilePath = os.path.join(nlabPath,filename)
        nLabs = imread(nFilePath)
        print("Reading: ",nFilePath)
        ncells = np.shape(np.unique(nLabs))[0]
        if verbose:
            print("Cell shapes=" + nLabs.shape) 
            print("#cells=",ncells)
        for rLabfname in rLabFiles:
            if rLabfname.endswith(filename_rna[10:]) and not filename.startswith("."):
                rLabPath = os.path.join(rLabFold,rLabfname)         
                rseg = imread(rLabPath)
                print("Reading: ", rLabPath)
                numRNA = np.max(rseg)
                lab,cnt = np.unique(rseg, return_counts=True)
                dfIsect = pd.DataFrame(index = range(0), columns = ["cellID","labelID","nPixinCell","nPixinback","nPix","inCfrac"])

# first get rna objects that overlap background, we will exclude these
                a0 = rseg[nLabs==0]
                lab0, cnt0 = np.unique(a0, return_counts=True)
        # objects that do not overlap background
                lab01 = np.setdiff1d(range(numRNA + 1),lab0)
                cnt01 = [0] * np.size(lab01)
        #combine (sorted to reflect original indexes)
                labx = np.concatenate((lab0,lab01))
                idx1 = np.argsort(labx)
                lab0 = labx[idx1]
                cnt0 = np.concatenate((cnt0,cnt01))[idx1]
                ix = -1
                for ic in np.unique(nLabs):
                    if ic==0: continue
                    ix=ix + 1
                    a = rseg[nLabs==ic]             # get the pixels in rseg, corresponding to the cell
                    lab1,cnt1 = np.unique(a, return_counts=True)
                    ocnt0 = cnt0[lab1]
                    ocnt = cnt[lab1]
                    inCfrac = cnt1/ocnt        # ratio in cell
                                       # keep the inc overlaps and the nonoverlaps
                    icx = np.repeat(ic,np.size(lab1))
# note, zero should be in val0, so should not be uval
                    if np.size(lab1) > 0:
                        nLabsPix = np.column_stack((icx,lab1,cnt1,ocnt0,ocnt,inCfrac))
                        tempdf = pd.DataFrame(data = nLabsPix, index = range(np.size(lab1)), columns = ["cellID","labelID","nPixinCell","nPixinback","nPix","inCfrac"])
                        tempdf = tempdf[tempdf['labelID']!=0]
                        dfIsect = pd.concat((dfIsect,tempdf))
                isectTabFile = rLabfname.replace("lab.tif","nuclei_intersections.csv")
                isectTabPath = os.path.join(tabFold,isectTabFile)
                dfIsect.to_csv(isectTabPath)
                print("saved intersections file")

exp20_nuc_021524_K913_B1_RHtop_T2b_K9_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/K913_nucquant/nuc/nLabs/upscaled_expanded/exp20_nuc_021524_K913_B1_RHtop_T2b_K9_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/K913_nucquant/rna/rLabs/rna1_021524_K913_B1_RHtop_T2b_K9_lab.tif


/var/folders/05/0lb791q54mq3py_0zyc6cp080000gp/T/ipykernel_16242/928118511.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dfIsect = pd.concat((dfIsect,tempdf))


saved intersections file
Reading:  /Volumes/RWK_Imaging/K913_nucquant/rna/rLabs/rna2_021524_K913_B1_RHtop_T2b_K9_lab.tif


/var/folders/05/0lb791q54mq3py_0zyc6cp080000gp/T/ipykernel_16242/928118511.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dfIsect = pd.concat((dfIsect,tempdf))


saved intersections file
exp20_nuc_112023_K913P20_A5_LH_40xstack_T2b_K9_fine_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/K913_nucquant/nuc/nLabs/upscaled_expanded/exp20_nuc_112023_K913P20_A5_LH_40xstack_T2b_K9_fine_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/K913_nucquant/rna/rLabs/rna1_112023_K913P20_A5_LH_40xstack_T2b_K9_fine_lab.tif


...

saved intersections file
exp20_nuc_022624_K913_A4_RH_T2b_K9_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/K913_nucquant/nuc/nLabs/upscaled_expanded/exp20_nuc_022624_K913_A4_RH_T2b_K9_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/K913_nucquant/rna/rLabs/rna1_022624_K913_A4_RH_T2b_K9_lab.tif


/var/folders/05/0lb791q54mq3py_0zyc6cp080000gp/T/ipykernel_16242/928118511.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dfIsect = pd.concat((dfIsect,tempdf))


saved intersections file
Reading:  /Volumes/RWK_Imaging/K913_nucquant/rna/rLabs/rna2_022624_K913_A4_RH_T2b_K9_lab.tif


/var/folders/05/0lb791q54mq3py_0zyc6cp080000gp/T/ipykernel_16242/928118511.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dfIsect = pd.concat((dfIsect,tempdf))


saved intersections file
